# FHIR Query Validator Factory - Loop Engineering Demo

This notebook demonstrates the feedback loops implemented in the system:

1. **Normal valid query** — cache → validate → execute
2. **Learner escalation** — repeated invalid queries (3+ in 10 minutes)
3. **Server switcher** — `hapi`, `firely`, and **mock.health** (`mockhealth`, requires `MOCK_HEALTH_API_KEY` in `.env.local`)
4. **Human escalation** — repeated invalid queries (5+ in 15 minutes) with pause/resume

In [1]:
import sys
import os

sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))))

from src.agentic_layer.graph.validation_workflow import run_validation_workflow
from src.agentic_layer.graph.workflow_engine import human_gate
from src.agentic_layer.config.settings import DEFAULT_SERVERS


def print_summary(result: dict, *, label: str = "") -> None:
    """Print a compact view of workflow final_output."""
    out = result.get("final_output", {})
    if label:
        print(f"\n--- {label} ---")
    print(f"  server_used           : {out.get('server_used')}")
    print(f"  valid                 : {out.get('valid')}")
    print(f"  executed              : {out.get('executed')}")
    print(f"  pattern_detected      : {out.get('pattern_detected')}")
    print(f"  escalation            : {out.get('escalation')}")
    print(f"  human_review_required : {out.get('human_review_required')}")
    if out.get("errors"):
        print(f"  errors                : {out['errors'][:2]}")
    if result.get("learner_guidance"):
        print(f"  learner_message       : {result['learner_guidance'].get('message')}")
    if out.get("human_review"):
        review = out["human_review"]
        print(f"  review_id             : {review.get('review_id')}")
        print(f"  severity              : {review.get('severity')}")

## Scenario 1: Normal Valid Query

In [2]:
state = {
    "query_url": "Patient?gender=male",
    "server_key": "hapi",
    "user_id": "user-alice",
    "mode": "validate_and_execute",
}

result = run_validation_workflow(state)
print_summary(result, label="Scenario 1")


=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache MISS for 'hapi' — fetching from https://hapi.fhir.org/baseR4
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?gender=male

=== [LOOP] Validation → Execution Loop ===
[QueryExecution] Executing query on HAPI FHIR: https://hapi.fhir.org/baseR4/Patient?gender=male

=== [LOOP] Pattern Detection → Learning / Human Escalation ===

--- Scenario 1 ---
  server_used           : hapi
  valid                 : True
  executed              : True
  pattern_detected      : False
  escalation            : none
  human_review_required : False


## Scenario 2: Repeated Invalid Queries (Triggers Learner)

Threshold: **3+ invalid queries within 10 minutes** → `escalation: learner`

In [3]:
for i in range(1, 4):
    print(f"\n--- Attempt {i} ---")
    state = {
        "query_url": "Patient?invalid_param=true",
        "server_key": "hapi",
        "user_id": "user-bob",
        "mode": "validate_and_execute",
    }
    result = run_validation_workflow(state)
    print_summary(result)


--- Attempt 1 ---

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'hapi' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?invalid_param=true

=== [LOOP] Pattern Detection → Learning / Human Escalation ===
  server_used           : hapi
  valid                 : False
  executed              : False
  pattern_detected      : False
  escalation            : none
  human_review_required : False
  errors                : ["Search parameter 'invalid_param' is not supported for Patient."]

--- Attempt 2 ---

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'hapi' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?invalid_param=true

=== [LOOP] Pattern Detection → Learning / Human Escalation ===
  server_used      

## Scenario 3: Server Switcher

Run the same query against each demo server. Each has a different `CapabilityStatement` (or auth requirements), so validation outcomes may differ.

| `server_key` | Server | Auth |
|--------------|--------|------|
| `hapi` | HAPI FHIR (public) | None |
| `firely` | Firely (public) | None |
| `mockhealth` | [mock.health](https://mock.health/docs) synthetic sandbox | Bearer API key — set `MOCK_HEALTH_API_KEY` in `.env.local` |

Copy `.env.example` to `.env.local` and add your mock.health key before running the `mockhealth` row.

In [11]:
QUERY = "Patient?gender=male"
DEMO_SERVERS = ["hapi", "firely","mockhealth"]
if os.getenv("MOCK_HEALTH_API_KEY"):
    DEMO_SERVERS.append("mockhealth")
else:
    print("Note: mockhealth skipped — set MOCK_HEALTH_API_KEY in .env.local to include mock.health\n")

print("Configured servers for this run:")
for key in DEMO_SERVERS:
    info = DEFAULT_SERVERS[key]
    auth = "Bearer API key" if info.get("requires_auth") else "none"
    print(f"  {key:12} → {info['name']} ({info['base_url']}) [auth: {auth}]")

print("\nRunning validate_only across servers (live HTTP to /metadata):")
for server_key in DEMO_SERVERS:
    result = run_validation_workflow({
        "query_url": QUERY,
        "server_key": server_key,
        "user_id": "notebook-server-switcher",
        "mode": "validate_only",
    })
    print_summary(result, label=server_key)

Note: mockhealth skipped — set MOCK_HEALTH_API_KEY in .env.local to include mock.health

Configured servers for this run:
  hapi         → HAPI FHIR (https://hapi.fhir.org/baseR4) [auth: none]
  firely       → Firely (https://server.fire.ly/R4) [auth: none]
  mockhealth   → mock.health (https://api.mock.health/fhir) [auth: Bearer API key]

Running validate_only across servers (live HTTP to /metadata):

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'hapi' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?gender=male

=== [LOOP] Pattern Detection → Learning / Human Escalation ===

--- hapi ---
  server_used           : hapi
  valid                 : True
  executed              : False
  pattern_detected      : False
  escalation            : none
  human_review_required : False

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH fo

In [5]:
# Execute on firely (public) and mock.health (when API key is configured)
for execute_key in ["firely"] + (["mockhealth"] if os.getenv("MOCK_HEALTH_API_KEY") else []):
    execute_query = "Patient?_count=1" if execute_key == "mockhealth" else QUERY
    result = run_validation_workflow({
        "query_url": execute_query,
        "server_key": execute_key,
        "user_id": "notebook-server-switcher",
        "mode": "validate_and_execute",
    })
    print_summary(result, label=f"{execute_key} validate_and_execute")


=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'firely' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?gender=male

=== [LOOP] Validation → Execution Loop ===
[QueryExecution] Executing query on Firely: https://server.fire.ly/R4/Patient?gender=male

=== [LOOP] Pattern Detection → Learning / Human Escalation ===

--- firely validate_and_execute ---
  server_used           : firely
  valid                 : True
  executed              : True
  pattern_detected      : False
  escalation            : none
  human_review_required : False


## Scenario 4: Human Escalation

Threshold: **5+ invalid queries within 15 minutes** → `escalation: human` and automated processing is **paused** for that user until a reviewer resolves the case.

In [6]:
HUMAN_USER = "user-charlie-human"
INVALID_QUERY = "Patient?invalid_param=true"

last_result = None
for i in range(1, 6):
    print(f"\n--- Human escalation attempt {i} ---")
    last_result = run_validation_workflow({
        "query_url": INVALID_QUERY,
        "server_key": "hapi",
        "user_id": HUMAN_USER,
        "mode": "validate_only",
    })
    print_summary(last_result)

assert last_result["final_output"]["escalation"] == "human"
assert last_result["final_output"]["human_review_required"] is True
print("\n✓ Human escalation triggered as expected.")


--- Human escalation attempt 1 ---

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'hapi' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?invalid_param=true

=== [LOOP] Pattern Detection → Learning / Human Escalation ===
  server_used           : hapi
  valid                 : False
  executed              : False
  pattern_detected      : False
  escalation            : none
  human_review_required : False
  errors                : ["Search parameter 'invalid_param' is not supported for Patient."]

--- Human escalation attempt 2 ---

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'hapi' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?invalid_param=true

=== [LOOP] Pattern Detection → Learning / Human 

In [7]:
# While paused, new requests for the same user are blocked
blocked = run_validation_workflow({
    "query_url": "Patient?gender=male",
    "server_key": "hapi",
    "user_id": HUMAN_USER,
    "mode": "validate_only",
})
print_summary(blocked, label="blocked while paused")
assert blocked["final_output"]["valid"] is False
assert "paused" in blocked["final_output"]["errors"][0].lower()


--- blocked while paused ---
  server_used           : hapi
  valid                 : False
  executed              : False
  pattern_detected      : None
  escalation            : None
  human_review_required : None
  errors                : ["User 'user-charlie-human' is paused pending human review."]


In [8]:
# Resolve the human review and resume automated processing
review_id = last_result["final_output"]["human_review"]["review_id"]

resolution = human_gate.submit_review_decision(
    review_id,
    reviewer="notebook-operator",
    decision="continue_monitoring",
    rationale="Demo resolution — allow user to continue with monitoring.",
)
print(f"Review resolved: {resolution['review']['decision']}")
print(f"User resumed   : {resolution['resumed']}")

resumed = run_validation_workflow({
    "query_url": "Patient?gender=male",
    "server_key": "hapi",
    "user_id": HUMAN_USER,
    "mode": "validate_only",
})
print_summary(resumed, label="after human resolution")
assert resumed["final_output"]["valid"] is True

[HumanInterventionGate] Review 82ce44c2-69fc-45fb-ba9b-ba91b2657c87 resolved: continue_monitoring
Review resolved: continue_monitoring
User resumed   : True

=== [LOOP] Cache Invalidation Loop ===
[CacheAgent] Cache REFRESH for 'hapi' (conditional miss)
[CapabilityInterpreter] Interpreting CapabilityStatement...

=== [LOOP] Validation + Pattern Detection ===
[QueryValidator] Validating query: Patient?gender=male

=== [LOOP] Pattern Detection → Learning / Human Escalation ===

--- after human resolution ---
  server_used           : hapi
  valid                 : True
  executed              : False
  pattern_detected      : False
  escalation            : none
  human_review_required : False
